# Phase 34 — Computational Efficiency & Runtime Profiling
## v1.1 Runtime-Semantics + Analytical-Comparator Reconciliation (Reviewer Notebook)

**Classification:** `READ_ONLY_PHASE34_V1_1_RUNTIME_SEMANTICS_RECONCILIATION_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase34_efficiency_profiling_v1_1`

v1.0 was pre-execution and produced **no measurements**. v1.1 repairs two specification-level
issues before the first measurement. No measured Phase34 result changed, because none existed.


## 0. Setup

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path(r"D:\Softfors\jup")
AUD = ROOT / "reports" / "audits"
TAB = ROOT / "reports" / "tables"


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def tshow(name, n=None):
    df = pd.read_csv(TAB / name)
    print(name, "| shape:", df.shape)
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None, "display.width", 240):
        print(df.to_string(index=False))
    return df


print("Phase34 v1.1 runtime-semantics + analytical-comparator reconciliation")
print("READ-ONLY. No model load, no forward, no CUDA timing, no memory profiling,")
print("no profiler tools, no protected data, no TEST, no training.")


Phase34 v1.1 runtime-semantics + analytical-comparator reconciliation
READ-ONLY. No model load, no forward, no CUDA timing, no memory profiling,
no profiler tools, no protected data, no TEST, no training.


## 1. v1.0 preservation and defect classification

In [2]:
dr = jload(AUD / "phase34_v1_0_runtime_semantics_and_comparator_defect_record.json")
print("classification:", dr["classification"])
for i in dr["issues"]:
    print(f"  {i['issue_id']}: {i['code']}")
print()
print("model measurements already performed:", dr["model_measurements_already_performed"])
print("scientific runtime results exist:", dr["scientific_runtime_results_exist"])
print("latency results exist:", dr["latency_results_exist"])
print("memory results exist:", dr["memory_results_exist"])
print("runtime results invalidated:", dr["runtime_results_invalidated"])
print("repair required before first measurement:", dr["repair_required_before_first_measurement"])
print("v1.0 artifacts preserved:", dr["v1_0_artifacts_preserved"])
assert dr["model_measurements_already_performed"] is False


classification: PRE_EXECUTION_SPECIFICATION_SEMANTICS_REPAIR__NO_MEASURED_RESULTS_EXIST
  ISSUE 1: UNIFORM_FP32_RUNTIME_PROTOCOL_DOES_NOT_MATCH_CANONICAL_COMPONENT_SPECIFIC_INFERENCE_SEMANTICS
  ISSUE 2: LITERAL_ALWAYS_REFINE_PREDECESSOR_COMPARATOR_CONFLATED_WITH_GATED_EXECUTE_ALL_GRAPH

model measurements already performed: False
scientific runtime results exist: False
latency results exist: False
memory results exist: False
runtime results invalidated: False
repair required before first measurement: True
v1.0 artifacts preserved: True


## 2. Issue 1 — canonical component-specific runtime precision

G is the **only** component whose canonical inference runs under `autocast(float16)` with
`torch.no_grad()` and `inference_mode = false`. R_psi / B_phi / S_omega run with autocast
disabled under `torch.inference_mode()`.

In [3]:
pc = jload(AUD / "phase34_precision_execution_semantics_contract_v1_1.json")
print("primary runtime profile:", pc["primary_runtime_profile"])
print("secondary uniform FP32 profile:", pc["secondary_uniform_fp32_profile"])
print("component semantics not interchangeable:", pc["component_semantics_not_interchangeable"])
print()
for cid, s in pc["PRIMARY_CANONICAL_COMPONENT_SEMANTICS"].items():
    print(f"  {cid:8s} dtype={s['model_dtype']:8s} autocast={str(s['autocast_enabled']):5s} "
          f"dtype={s['autocast_dtype']} ctx={s['grad_context']:22s} "
          f"inference_mode={s['inference_mode']} sci_batch={s['canonical_scientific_batch_semantics']}")
print()
print("invariants:")
for inv in pc["invariants"]:
    print("  -", inv)


primary runtime profile: COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS
secondary uniform FP32 profile: INELIGIBLE_NOT_REQUIRED
component semantics not interchangeable: True

  G        dtype=float32  autocast=True  dtype=float16 ctx=torch.no_grad()        inference_mode=False sci_batch=batch 8, contiguous chunks of 8 inside a single domain sorted by (dataset, sample_id); batch-membership SENSITIVE under autocast fp16
  R_psi    dtype=float32  autocast=False dtype=None ctx=torch.inference_mode() inference_mode=True sci_batch=batch 8
  B_phi    dtype=float32  autocast=False dtype=None ctx=torch.inference_mode() inference_mode=True sci_batch=FULL_MATRIX_SINGLE_FORWARD
  S_omega  dtype=float32  autocast=False dtype=None ctx=torch.inference_mode() inference_mode=True sci_batch=batch 128

invariants:
  - G_AUTOCAST_SEMANTICS_MUST_NOT_BE_APPLIED_TO_R_PSI_B_PHI_OR_S_OMEGA
  - UNIFORM_FP32_NO_AUTOCAST_MUST_NOT_REPLACE_G_CANONICAL_RUNTIME_SEMANTICS


## 3. Batch-size semantics

Batch 1 remains the primary latency measurement but is **not** canonical numerical
reproduction: the frozen scientific G path uses batch 8 and is batch-membership sensitive
under autocast(fp16).

In [4]:
print("batch1 label:")
print(" ", pc["PRIMARY_CANONICAL_COMPONENT_SEMANTICS"]["G"]["phase34_batch1_benchmark_semantics"])
sc = jload(AUD / "phase34_measured_runtime_scope_v1_1.json")
print()
print("timing-scope classifications:")
for arm, v in sc["per_arm_timing_scope"].items():
    print(f"  {arm:22s} -> {v['classification']}")
print()
print("END_TO_END_PIPELINE_RUNTIME used:", sc["end_to_end_pipeline_runtime_used"])


batch1 label:
  batch 1 hardware microbenchmark under the canonical G precision/autocast/gradient context; batch membership differs and is disclosed

timing-scope classifications:
  GLOBAL_ONLY_C1         -> MODEL_COMPUTE_PATH_RUNTIME
  FINAL_ROUTED_A8_C9     -> MODEL_COMPUTE_PATH_RUNTIME
  G_COMPONENT            -> SYNTHETIC_COMPONENT_RUNTIME
  R_PSI_COMPONENT        -> SYNTHETIC_COMPONENT_RUNTIME
  B_PHI_COMPONENT        -> SYNTHETIC_COMPONENT_RUNTIME
  S_OMEGA_COMPONENT      -> SYNTHETIC_COMPONENT_RUNTIME

END_TO_END_PIPELINE_RUNTIME used: False


## 4. Issue 2 — analytical comparators reconciled

`LITERAL_ALWAYS_REFINE_C2` **bypasses routing** (`router_present = false`). A gated
execute-all C9 graph retains router overhead and is labelled separately.

In [5]:
ac = jload(AUD / "phase34_analytical_compute_contract_v1_1.json")
print("label:", ac["label"])
print()
for name, c in ac["comparators"].items():
    print(f"  {name}")
    print(f"      formula = {c['formula']}")
    print(f"      MAC     = {c['mac']}")
    print(f"      router={c['router_present']} refiner={c['refiner_present']} "
          f"s_omega={c['s_omega_present']}")
    print(f"      label   = {c['scientific_label']}")
print()
print("C2 router present:", ac["literal_always_refine_C2_router_present"])
print("C2 != gated execute-all C9:", ac["literal_always_refine_C2_not_equal_gated_execute_all_C9"])
print("denominators merged:", ac["comparator_denominators_merged"])


label: ANALYTICAL MAC LOWER BOUND

  literal_global_only_C1
      formula = MAC(G)
      MAC     = 31551740928
      router=False refiner=False s_omega=False
      label   = LITERAL_GLOBAL_ONLY_C1__ANALYTICAL_MAC_LOWER_BOUND
  literal_always_refine_C2
      formula = MAC(G) + MAC(R_psi)
      MAC     = 34782403584
      router=False refiner=True s_omega=False
      label   = LITERAL_ALWAYS_REFINE_C2__NO_ROUTING__ANALYTICAL_MAC_LOWER_BOUND
  final_A8_C9_expected_conditional
      formula = MAC(G) + MAC(B_phi) + p_activate * (MAC(R_psi) + MAC(S_omega))
      MAC     = 32437902022.582855
      router=True refiner=True s_omega=True
      label   = ANALYTICAL_MAC_LOWER_BOUND
  optional_gated_execute_all_C9
      formula = MAC(G) + MAC(B_phi) + MAC(R_psi) + MAC(S_omega)
      MAC     = 34782440912
      router=True refiner=True s_omega=True
      label   = GATED_EXECUTE_ALL_C9_100PCT_ACTIVATION_REFERENCE__NOT_LITERAL_C2
  phase25e_gated_K_N_eligible_pre_gate
      formula = MAC(G) + MAC(B_ph

## 5. Static comparator arithmetic (no model, no profiler)

In [6]:
r = ac["ratios"]
print("ratio vs LITERAL_GLOBAL_ONLY_C1        :", r["ratio_vs_LITERAL_GLOBAL_ONLY_C1"])
print("ratio vs LITERAL_ALWAYS_REFINE_C2      :", r["ratio_vs_LITERAL_ALWAYS_REFINE_C2"])
print("saving vs LITERAL_ALWAYS_REFINE_C2     :",
      r["absolute_saving_vs_LITERAL_ALWAYS_REFINE_C2"], "MACs")
print("ratio vs gated execute-all C9          :", r["ratio_vs_gated_execute_all_C9"])
print("saving vs gated execute-all C9         :",
      r["absolute_saving_vs_gated_execute_all_C9"], "MACs")
print()
C1 = ac["comparators"]["literal_global_only_C1"]["mac"]
C2 = ac["comparators"]["literal_always_refine_C2"]["mac"]
E = ac["comparators"]["final_A8_C9_expected_conditional"]["mac"]
assert C1 == 31551740928 and C2 == 34782403584
assert abs(r["ratio_vs_LITERAL_ALWAYS_REFINE_C2"] - E / C2) < 1e-12
print("assertions OK: C1/C2 exact; ratio reproduced from frozen integers")


ratio vs LITERAL_GLOBAL_ONLY_C1        : 1.0280859651011032
ratio vs LITERAL_ALWAYS_REFINE_C2      : 0.9325951826257453
saving vs LITERAL_ALWAYS_REFINE_C2     : 2344501561.417145 MACs
ratio vs gated execute-all C9          : 0.9325941817784192
saving vs gated execute-all C9         : 2344538889.417145 MACs

assertions OK: C1/C2 exact; ratio reproduced from frozen integers


## 6. v1.1 profiling arm registry

In [7]:
tshow("phase34_profiling_arm_registry_v1_1.csv")

phase34_profiling_arm_registry_v1_1.csv | shape: (8, 13)
                                        arm_id                  predecessor_authority                                                     scientific_meaning   components_executed  router_present  refiner_present  s_omega_present  runtime_capable                        runtime_precision_profile                                                                           batch1_label                            batch8_label  canonical_numerical_reproduction        final_eligibility
                                GLOBAL_ONLY_C1             Phase25E literal C1 anchor                                   global model alone; routing bypassed                     G           False            False            False             True COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH1_NOT_CANONICAL_G_NUMERICAL_REPRODUCTION SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK                             False       

,arm_id,predecessor_authority,scientific_meaning,components_executed,router_present,refiner_present,s_omega_present,runtime_capable,runtime_precision_profile,batch1_label,batch8_label,canonical_numerical_reproduction,final_eligibility
0,GLOBAL_ONLY_C1,Phase25E literal C1 anchor,global model alone; routing bypassed,G,False,False,False,True,COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS,CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH...,SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK,False,ELIGIBLE
1,FINAL_ROUTED_A8_C9,Phase26 canonical bundle A8/C9,"full routed system: G always, B_phi pre-gate, ...","G,B_phi,R_psi,S_omega",True,True,True,True,COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS,CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH...,SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK,False,ELIGIBLE
2,LITERAL_ALWAYS_REFINE_C2_ANALYTICAL_ONLY,Phase25E literal C2 no-routing anchor,analytical always-refine reference; routing by...,"G,R_psi",False,True,False,False,ANALYTICAL_ONLY,NaN,NaN,False,ELIGIBLE_ANALYTICAL_ONLY
3,GATED_EXECUTE_ALL_C9_IF_PREDECESSOR_AUTHORIZED,"Phase26 state machine, 100% activation",gated execute-all graph with router and post-s...,"G,B_phi,R_psi,S_omega",True,True,True,False,ANALYTICAL_ONLY,NaN,NaN,False,ELIGIBLE_ANALYTICAL_ONLY
4,G_COMPONENT,Phase26 component inventory,component-only G,G,False,False,False,True,COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS,CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH...,SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK,False,ELIGIBLE
5,R_PSI_COMPONENT,Phase25A resource accounting,component-only R_psi,R_psi,False,True,False,True,COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS,CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH...,SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK,False,ELIGIBLE
6,B_PHI_COMPONENT,Phase25D router,component-only B_phi,B_phi,True,False,False,True,COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS,CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH...,SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK,False,ELIGIBLE
7,S_OMEGA_COMPONENT,Phase25F post-safety,component-only S_omega,S_omega,False,False,True,True,COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS,CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH...,SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK,False,ELIGIBLE


## 7. v1.1 source freeze

In [8]:
sf = jload(AUD / "phase34_preexecution_source_freeze_v1_1.json")
for s in sf["frozen_sources"]:
    h = s["sha256"][:16] if s["sha256"] else "MISSING"
    print(f"  {h}...  {s['role']:52s} {s['path']}")
print()
print("v1.0 source freeze preserved:", sf["v1_0_source_freeze_preserved"])
print("v1.1 supersedes v1.0 for future measurement:",
      sf["v1_1_supersedes_v1_0_for_future_measurement"])
print("no v1.0 measurement executed:", sf["no_v1_0_measurement_was_executed"])


  e6effd91ef847d27...  V1_1_FUTURE_EXECUTION_HARNESS                        scripts/analysis/phase34_efficiency_profiling_execution_v1_1.py
  bd2cd11a74489128...  V1_1_ISOLATED_PER_ARM_WORKER                         scripts/analysis/phase34_efficiency_profiling_worker_v1_1.py
  c03a1da7ecd7b3c9...  V1_1_SPECIFICATION_GENERATOR                         scripts/analysis/phase34_specification_v1_1.py
  5fbb48e8ad362153...  V1_1_REVIEWER_NOTEBOOK_BUILDER__REPORTING_ONLY       scripts/analysis/build_phase34_spec_v1_1_reviewer_notebook.py
  342f2163690c2fa8...  CANONICAL_WRAPPER_SOURCE                             src/models/barr/ra_medfilm_barr.py
  4fe0abde3ec3a183...  R_psi_SOURCE                                         src/models/barr/regional_refiner.py
  f8dab77d2dc01bd8...  B_phi_SOURCE                                         src/models/barr/router_variants.py
  a04ff3e61b4d900f...  G_SOURCE                                             src/models/film/global_medfilm.py

v1.0 source freez

## 8. v1.1 execution eligibility

In [9]:
ea = jload(AUD / "phase34_execution_eligibility_adjudication_v1_1.json")
for k, v in ea["families"].items():
    print(f"  {k:56s} = {str(v['value']):5s}")
print()
print("canonical component semantics runtime ready:",
      ea["canonical_component_semantics_runtime_ready"])
print("batch1 role:", ea["batch1_runtime_role"])
print("batch8 role:", ea["batch8_runtime_role"])
print("literal C2 reconciled:", ea["literal_C2_comparator_semantics_reconciled"])
print("measured runtime execution eligible:", ea["measured_runtime_execution_eligible"])


  historical_analytical_MAC_reuse_eligible                 = True 
  new_parameter_count_execution_eligible                   = True 
  new_analytical_MAC_profiler_execution_eligible           = False
  GPU_batch1_latency_execution_eligible                    = True 
  GPU_batch8_throughput_execution_eligible                 = True 
  GPU_memory_execution_eligible                            = True 
  component_level_runtime_execution_eligible               = True 
  path_level_runtime_execution_eligible                    = True 
  conditional_weighted_runtime_estimate_eligible           = True 
  synthetic_route_replay_eligible                          = False
  cold_start_timing_eligible                               = False
  raw_ROUTER_HOLDOUT_execution_eligible                    = False
  raw_GT_execution_eligible                                = False
  TEST_execution_eligible                                  = False
  training_eligible                                        = F

## 9. Zero-execution audit

In [10]:
ne = jload(AUD / "phase34_v1_1_no_execution_audit.json")
for k, v in ne["counters"].items():
    print(f"  {k:46s} = {v}")
print("all zero:", ne["all_zero"])
assert ne["all_zero"]


  model_loads                                    = 0
  checkpoint_deserializations_into_model_objects = 0
  model_forwards                                 = 0
  synthetic_forwards                             = 0
  latency_iterations                             = 0
  fps_measurements                               = 0
  gpu_inference_memory_measurements              = 0
  mac_flop_profiler_executions                   = 0
  raw_image_reads                                = 0
  raw_gt_reads                                   = 0
  router_holdout_raw_reads                       = 0
  test_reads                                     = 0
  training_steps                                 = 0
all zero: True


## 10. v1.1 final check and terminal

In [11]:
fc = jload(AUD / "phase34_specification_final_check_v1_1.json")
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
term = jload(AUD / "phase34_specification_terminal_v1_1.json")
print()
print("terminal:")
print(" ", term["terminal"])


final check: 25 checks, 0 failed

terminal:
  PHASE34_EFFICIENCY_PROFILING_SPEC_V1_1_COMPLETE__CANONICAL_COMPONENT_SPECIFIC_RUNTIME_PRECISION_SEMANTICS_RESTORED__BATCH1_HARDWARE_MICROBENCHMARK_DISTINGUISHED_FROM_CANONICAL_NUMERICAL_REPRODUCTION__LITERAL_ALWAYS_REFINE_C2_AND_GATED_EXECUTE_ALL_COMPARATORS_RECONCILED__V1_1_SOURCE_FREEZE_COMPLETE__EXECUTION_ELIGIBLE_NOT_STARTED


---
## Summary

| item | value |
|---|---|
| v1.0 measurements produced | 0 (pre-execution) |
| G canonical runtime | float32 + autocast(fp16) + no_grad + inference_mode FALSE |
| R_psi / B_phi / S_omega | float32 + autocast OFF + inference_mode |
| primary runtime profile | `COMPONENT_SPECIFIC_CANONICAL_PRECISION_SEMANTICS` |
| uniform FP32 profile | `INELIGIBLE_NOT_REQUIRED` |
| batch-1 role | hardware microbenchmark, NOT canonical numerical reproduction |
| LITERAL C1 | MAC(G) |
| LITERAL C2 | MAC(G) + MAC(R_psi) — routing bypassed |
| gated execute-all C9 | separate label, router retained |
| model loads · forwards · timing · memory | 0 · 0 · 0 · 0 |
| TEST | `LOCKED_UNTIL_PHASE39` |
| Phase34 execution | NOT started |
| Phase35 | NOT started |
